## 4. Visualisations simples (25%)

Utilisons maintenant les données ordonnées pour créer quelques figures simples sur les données agrégées. Pour chacune des questions ci-dessous, vous devez faire un choix de figure approprié pour montrer la relation demandée. Il existe généralement plusieurs façons correctes de le faire - si votre figure raconte la bonne histoire, vous obtiendrez tous les points.

### Questions

1. Produisez une figure comparant les types de tirs de toutes les équipes (c.-à-d. agrégez simplement tous les tirs), pour une saison de votre choix. Superposez le nombre de buts au nombre de tirs. Quel semble être le type de tir le plus dangereux (le plus haut pourcentage de buts, pas le plus grand nombre de buts)? Le type de tir le plus courant? Pourquoi avez-vous choisi cette figure? Ajoutez cette figure et cette discussion à votre article de blog.

- Diagramme à bandes?

2. Quelle est la relation entre la distance à laquelle un tir a été effectué et la chance qu'il s'agisse d'un but? Produisez une figure pour chaque saison entre 2018-19 et 2020-21 pour répondre à cette question, et ajoutez-la à votre article de blog avec quelques phrases décrivant votre figure. Y a-t-il eu beaucoup de changements au cours de ces trois saisons? Pourquoi avez-vous choisi cette figure?

- Scatter plot + moyenne?

3. Combinez les informations des sections précédentes pour produire une figure qui montre le pourcentage de buts (100 × # buts / # tirs) en fonction à la fois de la distance par rapport au filet et de la catégorie de type de tir (vous pouvez choisir une seule saison de votre choix). Discutez brièvement de vos observations ; par ex., quels pourraient être les types de tirs les plus dangereux?

- 

4. [IFT6758 seulement] Mettez en place une interface de requêtes par LLM (avec des outils comme PandasAI, une approche prompt-to-code maison, ou des appels validés à vos propres fonctions Pandas) pour interroger votre ensemble de données ordonné en langage naturel. PandasAI vous permet de poser des questions sur les données à des LLM et intègre les LLM à d'autres composantes du pipeline, comme la visualisation de données et l'ingénierie de caractéristiques.
- Vous concevrez des prompts et des outils qui permettent au LLM de comprendre des questions sur le hockey, de les convertir en requêtes structurées et de récupérer les réponses dans votre ensemble de données de la LNH.

- Testez des requêtes en convertissant le langage naturel en code ou en appels de fonctions validés, par exemple : « Compare les pourcentages de buts sur tirs du poignet de Montréal et de Toronto en 2020-21 » en une requête que votre code Pandas peut exécuter. Privilégiez la réutilisation de vos fonctions d'analyse vérifiées. Si vous exécutez du code Python généré par un LLM, utilisez un environnement isolé, sans identifiants ni accès à des fichiers personnels.

- Évaluez votre système sur un ensemble fixe de 8 questions d'évaluation, rédigées avant de commencer à ajuster vos prompts et jamais utilisées pour le développement des prompts. Elles doivent couvrir le filtrage, le regroupement, les pourcentages de buts, les graphiques, et au moins une question à laquelle l'ensemble de données ne permet pas de répondre (ex. sur les passes, qui ne sont pas suivies). Comparez chaque résultat à une réponse de référence que vous avez vérifiée manuellement ; un graphique est correct s'il montre les quantités, le regroupement et la saison demandés. Pour les questions sans réponse possible, le système doit indiquer qu'il ne peut pas répondre plutôt qu'inventer un résultat. Présentez un tableau indiquant, pour chaque question : si la première tentative s'est exécutée avec succès, si la réponse/le graphique final est correct, et le nombre de nouvelles tentatives. Discutez d'un succès et d'un échec ou d'une limite ; si aucun échec ne survient, testez une requête supplémentaire plus difficile et discutez-en.


In [ ]:
import json
import os
from ift6758.data import load_season
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "data_science_project")
os.environ["NHL_DATA_DIR"] = str(ROOT / "ift6758" / "data" / "data" / "raw")

In [ ]:
# DATAFRAME TEMPORAIRE - EN ATTENTE DE LA VERSION FINALE POUR ENTRER LES BONNES DONNÉES
def game_shots_to_pandas(game:dict, season:int):
    """Extraire la liste de tirs et de buts d'une seule partie en dataframe pandas"""

    # Liste d'événements
    plays = game["plays"]

    # Conversion en dataframe
    events = pd.json_normalize(plays)

    # Garder chaque tir et chaque but
    keep = events["typeDescKey"].isin(["shot-on-goal", "goal"])
    shots = events.loc[keep].copy()

    # Game ID
    shots["game_id"] = game["id"]

    # Game type
    shots["game_type"] = game["gameType"]

    # Numéro de la saison
    shots["season"] = f"{season}-{season+1}"

    # Nommer les colonnes du dataframe
    column_names = {
        "game_id": "game_id",
        "eventId": "event_id",
        "season": "season",
        "game_type": "game_type",
        "periodDescriptor.number": "period",
        "periodDescriptor.periodType": "period_type",
        "timeInPeriod": "time_in_period",
        "situationCode": "situation",
        "typeDescKey": "event_type",
        "details.eventOwnerTeamId": "team_id",
        "details.xCoord": "x",
        "details.yCoord": "y",
        "details.zoneCode": "zone",
        "details.shotType": "shot_type",
        "details.shootingPlayerId": "shooter_id",
        "details.scoringPlayerId": "scorer_id",
        "details.goalieInNetId": "goalie_id"
    }

    # Renommer les colonnes
    shots = shots.reindex(columns=list(column_names)).rename(columns=column_names)

    # Noms des équipes
    team_names = {
        game["awayTeam"]["id"]: game["awayTeam"]["abbrev"],
        game["homeTeam"]["id"]: game["homeTeam"]["abbrev"],
    }
    shots["team"] = shots["team_id"].map(team_names)

    # Noms des joueurs
    """player_names = {}
    for i in range(40):
        first_name = shots["rosterSpots"][{i}]["firstName"]["default"]
        last_name = shots["rosterSpots"][{i}]["lastName"]["default"]
        name = f"{first_name} {last_name}"
        player_names["player_id"] = name
    if shots["shooter_id"]:
        shots["player_name"] = name
    else:
        shots[""]"""

    # Tir/but en supériorité numérique, infériorité numérique ou jeu normal
    """situationCode: 4 digits [gardien visiteur][patineurs visiteurs][patineurs locaux][gardien local]
    premier et dernier digit 0 ou 1 (absent ou présent) // digits du milieu: jusqu'à 6 si gardien au banc"""

    # Indicateur de but ou non
    shots["is_goal"] = shots["event_type"].eq("goal")

    # Indicateur de filet désert ou non
    shots["empty_net"] = shots["goalie_id"].isna()

    # Convertir en valeurs manquantes
    shots = shots.convert_dtypes()
    return shots

def season_shots_to_pandas(season: int):
    """Extraire le dataframe pandas de tous les matchs d'une saison"""
    whole_season = load_season(season, "regular") + load_season(season, "playoffs")

    # Liste de dataframes
    dataframes = []

    # Extraire chaque dataframe d'un match
    for game in whole_season:
        # Claude.AI utilisé, cf annexe
        try:
            df = game_shots_to_pandas(game, season)
        except Exception as e:
            print(f"Skipping game {game}: {e}")
            continue
        
        dataframes.append(df)

    # Soulever une erreur si aucun match n'est extrait
    if not dataframes:
        raise ValueError("Aucune partie trouvée.")

    # Retourner le dataframe avec tous les matchs d'une saison
    return pd.concat(dataframes, ignore_index=True)

    # Une fois la fonction complète, sauvegarder en CVS en remplaçant les chemins de la démo pour les bons:
    """Save the cleaned table to a CSV file.
    processed_dir = Path("data/processed")
    processed_dir.mkdir(parents=True, exist_ok=True)
    csv_path = processed_dir / f"{game_id}_shots.csv"
    shots.to_csv(csv_path, index=False)
    print("Saved processed table:", csv_path.resolve())"""

In [ ]:
# Question 1: Construire la figure, un diagramme à bandes
def get_shots_or_goals(df, critere: str, goal_or_not: bool, count: bool=False):
    """Séparer les tirs et les buts selon un critère, avec l'option de les compter ou non avec get_count"""
    keep = df["is_goal"] == goal_or_not
    if count == False:
        goals_or_shots = df.loc[keep].copy()
    else:
        goals_or_shots = df.loc[keep].copy()[critere].value_counts()
    
    return goals_or_shots

def create_Q1_figure(df):
    """Créer la première figure"""
    # Value_counts des tirs et des buts selon le critère "Type de tir"
    goal_count = get_shots_or_goals(df, critere="shot_type", goal_or_not=True, count=True)
    shot_count =  get_shots_or_goals(df, critere="shot_type", goal_or_not=False, count=True)

    # Transformer en array
    goals = goal_count.to_numpy()
    shots = shot_count.to_numpy()

    # Pourcentage de buts
    goal_percent = [100*goals[i]/(shots[i]+goals[i]) for i in range(0, len(shot_count.index))]

    # Nombre de colonnes déterminé par le nombre de labels pour chaque tir
    x = np.arange(len(shot_count.dropna()))
    width = 0.35

    # Figure
    fig, ax1 = plt.subplots(figsize=(9, 6))

    # Premier axe y: les tirs et les buts
    ax1.bar(x, shots, width, color="#0e2d9c")
    ax1.bar(x, goals, width, bottom=shots, color="#008f30")

    # Étiquettes dérivées des colonnes de type de tir
    ax1.set_xticks(x+0.1, shot_count.index)
    ax1.tick_params(length=0)
    ax1.set_ylabel("# de tirs")
    fig.suptitle(f"Tirs et buts par type de tir pendant la saison {df['season'][0]}", y = 0.93)
    fig.supxlabel("Type de tir", size = "medium", y=0.035)

    # Deuxième axe y: le pourcentage de buts
    ax2 = ax1.twinx()
    ax2.bar(x+0.3, goal_percent, width-0.2, color="#cc0202")
    ax2.set_ylabel("% buts")
    ax2.set_ylim(0, 30)
    fig.legend(["Tirs au but", "Buts", "% Buts"], loc='upper right', bbox_to_anchor=(0.88, 0.85))

    plt.show()

In [ ]:
create_Q1_figure(season_shots_to_pandas(2019))

In [ ]:
# Question 2
# En zone défensive, le filet est celui le plus loin.
# En zone neutre, nous assumons que le filet est celui le plus près.
def compute_distance(df):
    distances = []
    "Calculer la distance entre le tir et le filet et convertir en dataframe"
    for i in range(len(df)):
        try:
            x = df["x"][i]
            y = df["y"][i]

            # Filets à 89 et -89 pieds, y = 0
            distance_from_net1 = np.sqrt((x+89)**2+y**2)
            distance_from_net2 = np.sqrt((x-89)**2+y**2)

            # Distances arrondies pour agréger tous les tirs près les uns des autres
            if df["zone"][i] == "D":
                distance = max(distance_from_net1, distance_from_net2)
            else:
                distance = min(distance_from_net1, distance_from_net2)
            distances.append(int(distance))
            # Si la donnée est manquante, remplir quand même et continuer la boucle
        except TypeError:
            distance = np.nan
            distances.append(distance)
            continue
        
    
    df["distance"] = distances

def create_Q2_figure(df):
    "Figure qui représente la relation entre la distance au filet et la proportion de tirs qui sont des buts"
    compute_distance(df)
    df_distances = df[["distance", "is_goal"]].dropna()

    # Dictionnaire de buts et de tirs pour chaque distance
    goals_by_distance = {}
    shots_by_distance = {}

    #Pour chaque valeur de distance, créer une clé
    for i in df_distances["distance"]:
        goals_by_distance[int(i)]= 0
        shots_by_distance[int(i)]= 0

    # Pour chaque distance, ajouter le but ou le tir
    for distance, is_goal in zip(df_distances["distance"], df_distances["is_goal"]):
        key = int(distance)
        if is_goal:
            goals_by_distance[key] += 1
        else:
            shots_by_distance[key] += 1

    # Pourcentage de buts pour chaque distance
    goal_percent = []
    for i in sorted(goals_by_distance):
            goal_percent.append(100*goals_by_distance.get(i)/(shots_by_distance.get(i)+goals_by_distance.get(i)))
    
    # Axe horizontal = toutes les clés, qui équivalent aux distances
    x = sorted(goals_by_distance.keys())

    fig, ax = plt.subplots(figsize=(10, 6))
    max_possible_distance = np.sqrt(189**2 + 42.5**2)
    ax.set_xticks(np.arange(0, max_possible_distance, step=25))
    ax.tick_params(length=0)
    ax.set_ylabel("% de buts")
    ax.set_xlabel("Distance du filet (pi)")
    fig.suptitle(f"Proportion de buts par rapport au nombre de tirs selon la distance du filet pour la saison {df['season'][0]}", y = 0.93)

    plt.step(x, goal_percent, where='pre')
    plt.show()

In [ ]:
create_Q2_figure(season_shots_to_pandas(2018))
create_Q2_figure(season_shots_to_pandas(2019))
create_Q2_figure(season_shots_to_pandas(2020))